# Trading Portfolio Analysis — Data Collection Pipeline

This notebook builds the full dataset from scratch:

1. Download daily adjusted close prices for 110 tickers (11 sectors x 10 tickers) from Yahoo Finance
2. Reshape from wide (one column per ticker) to long (one row per Date/Ticker) format
3. Attach each ticker's sector and compute daily returns
4. Download the SPY benchmark and compute its daily return
5. Load everything into a local SQLite database
6. Run sanity checks to confirm nothing was lost/corrupted along the way

Every path below is **relative to the project root**, so this notebook runs the same
on any machine — no hardcoded `C:\Users\...` paths.


In [1]:
import pandas as pd
import yfinance as yf
import sqlite3
from pathlib import Path

# Project root = parent of this notebook's folder (notebooks/ -> project root)
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_DIR = PROJECT_ROOT / "data"
SQL_DIR = PROJECT_ROOT / "sql"
RAW_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

START_DATE = "2020-01-01"
BENCHMARK_TICKER = "SPY"

print("Project root:", PROJECT_ROOT)


Project root: /Users/palletsimonliquidations/projects/portfolio-analysis


/Users/palletsimonliquidations/projects/portfolio-analysis/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## Step 1 — Define the ticker universe

This is the 110-ticker, 11-sector universe from the existing project (10 tickers per sector,
GICS-style sector buckets). Keeping this as an explicit dict — rather than a bare list — means
the sector tag travels with the ticker instead of being bolted on later, which is where a lot
of the earlier version's confusion came from.

In [2]:
TICKER_SECTORS = {
    'TTWO': 'Communication Services', 'CHTR': 'Communication Services', 'META': 'Communication Services',
    'NFLX': 'Communication Services', 'DIS': 'Communication Services', 'T': 'Communication Services',
    'CMCSA': 'Communication Services', 'TMUS': 'Communication Services', 'VZ': 'Communication Services',
    'GOOGL': 'Communication Services',
    'AMZN': 'Consumer Discretionary', 'SBUX': 'Consumer Discretionary', 'ROST': 'Consumer Discretionary',
    'MCD': 'Consumer Discretionary', 'NKE': 'Consumer Discretionary', 'TSLA': 'Consumer Discretionary',
    'LOW': 'Consumer Discretionary', 'MAR': 'Consumer Discretionary', 'TJX': 'Consumer Discretionary',
    'HD': 'Consumer Discretionary',
    'KO': 'Consumer Staples', 'KR': 'Consumer Staples', 'COST': 'Consumer Staples', 'KMB': 'Consumer Staples',
    'MDLZ': 'Consumer Staples', 'PEP': 'Consumer Staples', 'PG': 'Consumer Staples', 'WMT': 'Consumer Staples',
    'CL': 'Consumer Staples', 'MO': 'Consumer Staples',
    'EOG': 'Energy', 'MPC': 'Energy', 'PSX': 'Energy', 'HAL': 'Energy', 'SLB': 'Energy', 'XOM': 'Energy',
    'CVX': 'Energy', 'COP': 'Energy', 'BKR': 'Energy', 'DVN': 'Energy',
    'AXP': 'Financials', 'SCHW': 'Financials', 'C': 'Financials', 'JPM': 'Financials', 'BLK': 'Financials',
    'GS': 'Financials', 'MS': 'Financials', 'BAC': 'Financials', 'USB': 'Financials', 'WFC': 'Financials',
    'MRK': 'Health Care', 'JNJ': 'Health Care', 'ABT': 'Health Care', 'TMO': 'Health Care', 'UNH': 'Health Care',
    'AMGN': 'Health Care', 'LLY': 'Health Care', 'PFE': 'Health Care', 'ABBV': 'Health Care', 'DHR': 'Health Care',
    'RTX': 'Industrials', 'BA': 'Industrials', 'MMM': 'Industrials', 'CAT': 'Industrials', 'LMT': 'Industrials',
    'EMR': 'Industrials', 'DE': 'Industrials', 'UPS': 'Industrials', 'GE': 'Industrials', 'FDX': 'Industrials',
    'CRM': 'Information Technology', 'AVGO': 'Information Technology', 'ORCL': 'Information Technology',
    'NVDA': 'Information Technology', 'AAPL': 'Information Technology', 'MSFT': 'Information Technology',
    'IBM': 'Information Technology', 'ADBE': 'Information Technology', 'AMD': 'Information Technology',
    'CSCO': 'Information Technology',
    'VMC': 'Materials', 'SHW': 'Materials', 'APD': 'Materials', 'ECL': 'Materials', 'FCX': 'Materials',
    'IP': 'Materials', 'NEM': 'Materials', 'MLM': 'Materials', 'PKG': 'Materials', 'LIN': 'Materials',
    'MAA': 'Real Estate', 'EQIX': 'Real Estate', 'WELL': 'Real Estate', 'PLD': 'Real Estate', 'DLR': 'Real Estate',
    'AMT': 'Real Estate', 'CCI': 'Real Estate', 'O': 'Real Estate', 'AVB': 'Real Estate', 'SPG': 'Real Estate',
    'ED': 'Utilities', 'DUK': 'Utilities', 'D': 'Utilities', 'AEP': 'Utilities', 'XEL': 'Utilities',
    'SRE': 'Utilities', 'SO': 'Utilities', 'PEG': 'Utilities', 'EXC': 'Utilities', 'NEE': 'Utilities',
}

TICKERS = sorted(TICKER_SECTORS.keys())
print(f"{len(TICKERS)} tickers across {len(set(TICKER_SECTORS.values()))} sectors")


110 tickers across 11 sectors


## Step 2 — Download raw prices

`yf.download` with a list of tickers returns a **wide** DataFrame with a MultiIndex column
(`(field, ticker)`). We only need the adjusted close, so we pull that one field out immediately
rather than carrying Open/High/Low/Volume around for the rest of the pipeline.

Note: `auto_adjust=True` (the current yfinance default) makes the plain `Close` column already
split/dividend-adjusted — that's what earlier versions of this pipeline were calling "Adj Close".

In [3]:
raw = yf.download(TICKERS, start=START_DATE, auto_adjust=True, progress=False)

# raw.columns is a MultiIndex like ('Close', 'AAPL'), ('Close', 'MSFT'), ...
close_wide = raw['Close'].copy()
close_wide.index.name = 'Date'

close_wide.to_csv(RAW_DIR / 'stock_data_raw.csv')
print(close_wide.shape)
close_wide.head()


(1668, 110)


Ticker,AAPL,ABBV,ABT,ADBE,AEP,AMD,AMGN,AMT,AMZN,APD,...,UNH,UPS,USB,VMC,VZ,WELL,WFC,WMT,XEL,XOM
Date,,,,,,,,,,,,,,,,,,,,,
2020-01-02,72.271553,68.210747,76.781113,334.429993,73.667068,49.099998,195.604980,189.909760,94.900497,197.348801,...,261.702209,87.781120,44.857128,134.808624,41.275196,66.201775,45.306541,36.203293,51.098160,52.606396
2020-01-03,71.568909,67.563286,75.845085,331.809998,73.588249,48.599998,194.277084,190.001129,93.748497,192.960236,...,259.053833,87.728500,44.334297,134.223083,40.835739,67.366524,45.028378,35.883698,51.343903,52.183472
2020-01-06,72.139183,68.096474,76.242447,333.709991,73.832619,48.389999,195.767929,189.951279,95.143997,192.874893,...,260.852142,87.337662,43.728123,133.986908,40.747845,68.390877,44.758659,35.810635,51.270180,52.584145
2020-01-07,71.799934,67.708000,75.818588,333.390015,73.848366,48.250000,193.926727,185.903732,95.343002,193.694519,...,259.277588,87.187340,43.311371,132.475586,40.294861,67.944809,44.387764,35.478863,51.163677,52.153793
2020-01-08,72.954903,68.187866,76.127663,337.869995,73.627686,47.830002,194.073380,187.516083,94.598503,194.744720,...,264.744202,87.683395,43.220440,133.533478,40.369236,67.961319,44.522636,35.357105,51.114536,51.367294


## Step 3 — Reshape to long format + attach sector

Long format (`Date, Ticker, Adj_Close`) is what the SQL table and Power BI both expect —
one row per observation, not one column per ticker. This is the step that was missing a clean
home in the old notebook (it existed, but only for the 12-ticker version).

In [4]:
long = (
    close_wide
    .reset_index()
    .melt(id_vars='Date', var_name='Ticker', value_name='Adj_Close')
    .sort_values(['Ticker','Date'])
    .reset_index(drop=True)
)

nulls = long[long['Adj_Close'].isna()]
print(f"{len(nulls)} rows with null Adj_Close")
if len(nulls):
    print(nulls.groupby('Ticker')['Date'].agg(['min', 'max', 'count']))

last_valid = long.dropna(subset=['Adj_Close']).groupby('Ticker')['Date'].max()
interior = nulls[nulls['Date'] < nulls['Ticker'].map(last_valid)]
assert interior.empty, f"Interior price gaps found — do not drop these:\n{interior}"

long = long.dropna(subset=['Adj_Close']).reset_index(drop=True)
# -----------------------------------------------------------------------

long['Sector'] = long['Ticker'].map(TICKER_SECTORS)
assert long['Sector'].notna().all(), "Ticker missing from TICKER_SECTORS"

print(long.shape)
long.head()


0 rows with null Adj_Close
(183480, 4)


,Date,Ticker,Adj_Close,Sector
0,2020-01-02,AAPL,72.271553,Information Technology
1,2020-01-03,AAPL,71.568909,Information Technology
2,2020-01-06,AAPL,72.139183,Information Technology
3,2020-01-07,AAPL,71.799934,Information Technology
4,2020-01-08,AAPL,72.954903,Information Technology


## Step 4 — Compute daily returns

Daily return per ticker, computed **within each ticker group** (never across the boundary
between two different tickers) using `groupby().pct_change()`. The first day for each ticker
is correctly left as `NaN` — there's no prior day to compare to.

In [5]:
long['Daily_return'] = long.groupby('Ticker')['Adj_Close'].pct_change(fill_method=None)

out_path = DATA_DIR / 'stock_data.csv'
long.to_csv(out_path, index=False)
print(f"Saved {len(long):,} rows to {out_path}")
long.head()


Saved 183,480 rows to /Users/palletsimonliquidations/projects/portfolio-analysis/data/stock_data.csv


,Date,Ticker,Adj_Close,Sector,Daily_return
0,2020-01-02,AAPL,72.271553,Information Technology,NaN
1,2020-01-03,AAPL,71.568909,Information Technology,-0.009722
2,2020-01-06,AAPL,72.139183,Information Technology,0.007968
3,2020-01-07,AAPL,71.799934,Information Technology,-0.004703
4,2020-01-08,AAPL,72.954903,Information Technology,0.016086


## Step 5 — Benchmark data (SPY)

Same treatment for the benchmark index, kept in its own file/table since it isn't part of the
110-ticker universe and shouldn't be joined in as if it were just another stock.

In [6]:
bench_raw = yf.download(BENCHMARK_TICKER, start=START_DATE, auto_adjust=True, progress=False)
bench = bench_raw['Close'].reset_index()
bench.columns = ['Date', 'Adj_Close']
bench['Ticker'] = BENCHMARK_TICKER
bench['Sector'] = None
bench['Benchmark_return'] = bench['Adj_Close'].pct_change(fill_method=None)
bench = bench[['Date', 'Ticker', 'Adj_Close', 'Sector', 'Benchmark_return']]

bench.to_csv(DATA_DIR / 'benchmark_data.csv', index=False)
print(f"Saved {len(bench):,} rows")
bench.head()


Saved 1,668 rows


,Date,Ticker,Adj_Close,Sector,Benchmark_return
0,2020-01-02,SPY,296.125366,None,NaN
1,2020-01-03,SPY,293.882935,None,-0.007573
2,2020-01-06,SPY,295.004150,None,0.003815
3,2020-01-07,SPY,294.174713,None,-0.002812
4,2020-01-08,SPY,295.742432,None,0.005329


## Step 6 — Load into SQLite

We use the schema in `sql/schema.sql` (SQLite-compatible — no `dbo.` prefixes, `IDENTITY`, or
`NVARCHAR`, which are SQL Server-only syntax) rather than hand-rolling `CREATE TABLE` here, so
the schema lives in one place and can be inspected/edited without touching this notebook.

In [7]:
db_path = DATA_DIR / 'stock_data.db'
conn = sqlite3.connect(db_path)

with open(SQL_DIR / 'schema.sql') as f:
    conn.executescript(f.read())

long_db = long.assign(Date=long['Date'].dt.strftime('%Y-%m-%d'))
bench_db = bench.assign(Date=bench['Date'].dt.strftime('%Y-%m-%d'))

long_db.to_sql('stock_data', conn, if_exists='append', index=False)
bench_db.to_sql('benchmark_data', conn, if_exists='append', index=False)

sector_dim = (
    long_db[['Ticker', 'Sector']]
    .drop_duplicates()
    .assign(Date=long_db['Date'].min())
    [['Date', 'Ticker', 'Sector']]
)
sector_dim.to_sql('sector_dim', conn, if_exists='append', index=False)

with open(SQL_DIR / 'analysis.sql') as f:
    conn.executescript(f.read())

conn.commit()
print("Loaded into", db_path)


Loaded into /Users/palletsimonliquidations/projects/portfolio-analysis/data/stock_data.db


## Step 7 — Sanity checks

Cheap checks that would have caught the earlier mismatches (12-ticker notebook vs. 110-ticker
CSV, partial SQLite load) immediately instead of silently sitting in the repo.

In [8]:
conn = sqlite3.connect(DATA_DIR / 'stock_data.db')

summary = pd.read_sql("""
    SELECT COUNT(*) AS rows, COUNT(DISTINCT Ticker) AS tickers,
           MIN(Date) AS first_date, MAX(Date) AS last_date
    FROM stock_data
""", conn)
print(summary.to_string(index=False))

# Every ticker present, and every ticker with the same number of trading days.
assert summary['tickers'][0] == len(TICKERS), "Ticker count doesn't match TICKER_SECTORS"
per_ticker = pd.read_sql("SELECT Ticker, COUNT(*) n FROM stock_data GROUP BY Ticker", conn)['n']
assert per_ticker.nunique() == 1, f"Ragged series — row counts differ: {per_ticker.value_counts().to_dict()}"
assert summary['rows'][0] == len(TICKERS) * per_ticker.iloc[0]

# No null prices survived the load, and exactly one null return per ticker
# (each ticker's first day, which has no prior close to compare against).
assert pd.read_sql("SELECT COUNT(*) c FROM stock_data WHERE Adj_Close IS NULL", conn)['c'][0] == 0
assert pd.read_sql("SELECT COUNT(*) c FROM stock_data WHERE Daily_return IS NULL", conn)['c'][0] == len(TICKERS)

# Benchmark covers the same trading calendar as the universe.
assert pd.read_sql("SELECT COUNT(*) c FROM benchmark_data", conn)['c'][0] == per_ticker.iloc[0]

# Dates stored as plain YYYY-MM-DD, matching the CSVs.
assert pd.read_sql("SELECT COUNT(*) c FROM stock_data WHERE Date NOT LIKE '____-__-__'", conn)['c'][0] == 0

# Both SQL files ran: 2 views from schema.sql + 3 from analysis.sql.
views = pd.read_sql("SELECT name FROM sqlite_master WHERE type='view' ORDER BY name", conn)['name'].tolist()
assert len(views) == 5, f"Expected 5 views, got {views}"
print("views:", views)

conn.close()
print("All checks passed.")


  rows  tickers first_date  last_date
183480      110 2020-01-02 2026-08-21
views: ['vw_benchmark_returns', 'vw_daily_returns', 'vw_monthly_returns', 'vw_monthly_volatility', 'vw_portfolio_overview']
All checks passed.
